# Expedia 10k Choice Model Comparison

This notebook compares the new `IMRCS_EM_K10` results against the existing Expedia methods stored in `record/`.

Files used:

- `record/10k_sample_expedia.txt`: SCM, SMNL, RCS, SeqINV, NSMNL, MNL, CMNL
- `record/10k_sample_expedia_mmnl.txt`: MMNL
- `record/10k_sample_expedia_imrcs_em_k10_unique.txt`: de-duplicated IMRCS EM results
- `record/10k_sample_expedia_smnl_k10.txt`: SMNL_K10 (matched K=10 SMNL baseline for IMRCS_EM_K10)
- `record/10k_sample_expedia_smnl_impatience_k10.txt`: SMNL_IMPATIENCE_K10 (SMNL + stage-patience, direct MLE)

Lower is better for negative log-likelihood, RMSE, weighted squared error, ChiSq100, MAPE, and time. Higher is better for hit@1/2/3.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path.cwd()
RECORD = ROOT / 'record' if (ROOT / 'record').exists() else ROOT
FIG_DIR = RECORD / 'expedia_imrcs_figures'
FIG_DIR.mkdir(exist_ok=True)

SITE_ORDER = [15, 14, 5, 24, 32]
MODEL_ORDER = ['SCM', 'SMNL', 'SMNL_K10', 'SMNL_IMPATIENCE_K10', 'RCS', 'SeqINV', 'NSMNL', 'MNL', 'CMNL', 'MMNL', 'IMRCS_EM_K10']
LOWER_BETTER = ['testNLL', 'rmse', 'weightedSq', 'chiSq100', 'mape', 'time']
HIGHER_BETTER = ['hit1', 'hit2', 'hit3']
METRICS = ['testNLL', 'rmse', 'weightedSq', 'chiSq100', 'mape', 'hit1', 'hit2', 'hit3', 'time']


## Load and Harmonize Results

The legacy files have no header and use the layout:

`model site trial testNLL rmse weightedSq mape hit1 hit2 hit3 time`

The IMRCS file has a header and additional EM diagnostics. We keep those diagnostics, while aligning common metrics with the legacy methods.


In [ ]:
legacy_cols = ['model', 'site', 'trial', 'testNLL', 'rmse', 'weightedSq', 'mape', 'hit1', 'hit2', 'hit3', 'time']

main = pd.read_csv(RECORD / '10k_sample_expedia.txt', sep=r'\s+', header=None, names=legacy_cols)
mmnl = pd.read_csv(RECORD / '10k_sample_expedia_mmnl.txt', sep=r'\s+', header=None, names=legacy_cols)
imrcs = pd.read_csv(RECORD / '10k_sample_expedia_imrcs_em_k10_unique.txt', sep=r'\s+')
smnl_k10 = pd.read_csv(RECORD / '10k_sample_expedia_smnl_k10.txt', sep=r'\s+')
smnl_imp = pd.read_csv(RECORD / '10k_sample_expedia_smnl_impatience_k10.txt', sep=r'\s+')

imrcs_aligned = imrcs.rename(columns={
    'testRMSE': 'rmse',
    'testWeightedSq': 'weightedSq',
    'testMAPE': 'mape',
}).copy()
imrcs_aligned = imrcs_aligned[[
    'model', 'site', 'trial', 'testNLL', 'rmse', 'weightedSq', 'mape',
    'hit1', 'hit2', 'hit3', 'time', 'trainNLL', 'testChiSq100',
    'purchaseRate', 'numIter', 'finalImprove', 'numRankings', 'numPatience',
    'trainSessions', 'testSessions'
]]

smnl_k10_aligned = smnl_k10.rename(columns={
    'testRMSE': 'rmse',
    'testWeightedSq': 'weightedSq',
    'testMAPE': 'mape',
}).copy()
smnl_k10_aligned = smnl_k10_aligned[[
    'model', 'site', 'trial', 'testNLL', 'rmse', 'weightedSq', 'mape',
    'hit1', 'hit2', 'hit3', 'time'
]]

smnl_imp_aligned = smnl_imp.rename(columns={
    'testRMSE': 'rmse',
    'testWeightedSq': 'weightedSq',
    'testMAPE': 'mape',
}).copy()
smnl_imp_aligned = smnl_imp_aligned[[
    'model', 'site', 'trial', 'testNLL', 'rmse', 'weightedSq', 'mape',
    'hit1', 'hit2', 'hit3', 'time'
]]

df = pd.concat([main, mmnl, imrcs_aligned, smnl_k10_aligned, smnl_imp_aligned], ignore_index=True, sort=False)
df['chiSq100'] = 100 * df['weightedSq']
df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
df['site'] = pd.Categorical(df['site'], categories=SITE_ORDER, ordered=True)
df['pair'] = list(zip(df['site'].astype(int), df['trial'].astype(int)))

# Use finite NLL for means/ranks. Keep the original testNLL column for diagnostics.
df['testNLL_finite'] = df['testNLL'].replace([np.inf, -np.inf], np.nan)

print('Rows by model:')
print(df.groupby('model', observed=True).size().reindex(MODEL_ORDER))
print()
print('Unique site/trial pairs by model:')
print(df.groupby('model', observed=True)['pair'].nunique().reindex(MODEL_ORDER))
print()
print('Non-finite testNLL counts:')
print((~np.isfinite(df['testNLL'])).groupby(df['model'], observed=True).sum().reindex(MODEL_ORDER).fillna(0).astype(int))


## Overall Means

For NLL, `Inf` values are excluded from the mean and standard deviation; the `nll_nonfinite` column reports how often this happened. In this run it only affects CMNL.


In [ ]:
summary_source = df.copy()
summary_source['testNLL'] = summary_source['testNLL_finite']
summary = summary_source.groupby('model', observed=True).agg(
    n=('trial', 'size'),
    nll_nonfinite=('testNLL_finite', lambda s: int(s.isna().sum())),
    testNLL_mean=('testNLL', 'mean'),
    testNLL_std=('testNLL', 'std'),
    rmse_mean=('rmse', 'mean'),
    rmse_std=('rmse', 'std'),
    chiSq100_mean=('chiSq100', 'mean'),
    chiSq100_std=('chiSq100', 'std'),
    mape_mean=('mape', 'mean'),
    mape_std=('mape', 'std'),
    hit1_mean=('hit1', 'mean'),
    hit1_std=('hit1', 'std'),
    hit2_mean=('hit2', 'mean'),
    hit2_std=('hit2', 'std'),
    hit3_mean=('hit3', 'mean'),
    hit3_std=('hit3', 'std'),
    time_mean=('time', 'mean'),
    time_std=('time', 'std'),
).reindex(MODEL_ORDER)

summary_rounded = summary.round({
    'testNLL_mean': 3, 'testNLL_std': 3,
    'rmse_mean': 6, 'rmse_std': 6,
    'chiSq100_mean': 6, 'chiSq100_std': 6,
    'mape_mean': 6, 'mape_std': 6,
    'hit1_mean': 6, 'hit1_std': 6,
    'hit2_mean': 6, 'hit2_std': 6,
    'hit3_mean': 6, 'hit3_std': 6,
    'time_mean': 3, 'time_std': 3,
})
summary_rounded.to_csv(RECORD / 'expedia_imrcs_comparison_summary.csv')
summary_rounded


## Average Ranking by Metric

In [ ]:
ranking_rows = []
for metric in ['testNLL_finite', 'rmse', 'chiSq100', 'mape', 'hit1', 'hit2', 'hit3', 'time']:
    display_metric = 'testNLL' if metric == 'testNLL_finite' else metric
    ascending = display_metric in LOWER_BETTER
    means = df.groupby('model', observed=True)[metric].mean().dropna().sort_values(ascending=ascending)
    for rank, (model, value) in enumerate(means.items(), start=1):
        ranking_rows.append({'metric': display_metric, 'rank': rank, 'model': model, 'mean': value})
rankings = pd.DataFrame(ranking_rows)
rankings.to_csv(RECORD / 'expedia_imrcs_metric_rankings.csv', index=False)
rankings.pivot(index='rank', columns='metric', values='model')


## Win Counts Across the 50 Site-Trial Instances

A win means the model is best on that specific `(site, trial)` pair for the metric. Non-finite NLL values are ignored for NLL wins.


In [ ]:
wins = []
for metric in ['testNLL_finite', 'rmse', 'chiSq100', 'mape', 'hit1', 'hit2', 'hit3']:
    display_metric = 'testNLL' if metric == 'testNLL_finite' else metric
    ascending = display_metric in LOWER_BETTER
    tmp = df.dropna(subset=[metric]).sort_values(metric, ascending=ascending)
    best = tmp.groupby(['site', 'trial'], observed=True).first().reset_index()
    counts = best['model'].value_counts().reindex(MODEL_ORDER).fillna(0).astype(int)
    for model, count in counts.items():
        wins.append({'metric': display_metric, 'model': model, 'wins': count})
win_table = pd.DataFrame(wins)
win_pivot = win_table.pivot(index='model', columns='metric', values='wins').reindex(MODEL_ORDER).fillna(0).astype(int)
win_pivot.to_csv(RECORD / 'expedia_imrcs_win_counts.csv')
win_pivot


## IMRCS Versus Selected Baselines

Positive values mean IMRCS improves over the baseline. For lower-is-better metrics, improvement is `(baseline - IMRCS) / baseline`. For hit metrics, improvement is `(IMRCS - baseline) / baseline`.


In [ ]:
mean_by_model = df.copy()
mean_by_model['testNLL'] = mean_by_model['testNLL_finite']
mean_by_model = mean_by_model.groupby('model', observed=True)[METRICS].mean()

baseline_models = ['RCS', 'SMNL', 'MMNL', 'NSMNL', 'SeqINV', 'MNL', 'CMNL', 'SCM']
rows = []
im = mean_by_model.loc['IMRCS_EM_K10']
for baseline in baseline_models:
    if baseline not in mean_by_model.index:
        continue
    b = mean_by_model.loc[baseline]
    row = {'baseline': baseline}
    for metric in ['testNLL', 'rmse', 'chiSq100', 'mape']:
        row[metric + '_improvement_pct'] = 100 * (b[metric] - im[metric]) / b[metric] if pd.notna(b[metric]) else np.nan
    for metric in ['hit1', 'hit2', 'hit3']:
        row[metric + '_improvement_pct'] = 100 * (im[metric] - b[metric]) / b[metric]
    row['time_ratio_imrcs_over_baseline'] = im['time'] / b['time']
    rows.append(row)
improve = pd.DataFrame(rows).set_index('baseline').round(3)
improve.to_csv(RECORD / 'expedia_imrcs_relative_improvement.csv')
improve


## Per-Site Means

In [ ]:
# Per-site means. Keep the FULL table so downstream cells (the per-site line
# plot) still see every model; display + export only the subset requested here.
# SHOW_MODELS = ['RCS', 'SeqINV', 'NSMNL', 'MNL', 'CMNL', 'IMRCS_EM_K10', 'SMNL_IMPATIENCE_K10']

SHOW_MODELS = ['RCS', 'SeqINV', 'NSMNL', 'IMRCS_EM_K10', 'SMNL_IMPATIENCE_K10']

per_site = df.copy()
per_site['testNLL'] = per_site['testNLL_finite']
per_site_summary = per_site.groupby(['site', 'model'], observed=True)[['testNLL', 'rmse', 'chiSq100', 'mape', 'hit1', 'hit2', 'hit3', 'time']].mean()
per_site_summary = per_site_summary.reindex(pd.MultiIndex.from_product([SITE_ORDER, MODEL_ORDER], names=['site', 'model']))
per_site_summary.round(4).to_csv(RECORD / 'expedia_imrcs_per_site_summary.csv')

per_site_show = per_site_summary.reindex(pd.MultiIndex.from_product([SITE_ORDER, SHOW_MODELS], names=['site', 'model']))
per_site_show.round(4).to_csv(RECORD / 'expedia_imrcs_per_site_summary_selected.csv')
per_site_show.round(4)

## Per-Site LaTeX Table

Render the per-site means as a LaTeX table (one `\multirow` block per site, four measures: RMSE, 1-hit, 2-hit, 3-hit). The best model in each row is bolded — minimum for RMSE, maximum for the hit rates. Edit `MODEL_LABELS` to control the column headers.

In [ ]:
def per_site_to_latex(summary, models, sites=SITE_ORDER, measures=None,
                      model_labels=None, decimals=4,
                      caption='A comparison of the predictive performance of the fitted choice models on the Expedia data',
                      label='Tab:compare_expedia', site_fmt='Site {}'):
    """Render a per-site (site, model) summary as a LaTeX table, bolding the best
    model in each row. `measures` is a list of (column, header, higher_is_better)."""
    if measures is None:
        measures = [('rmse', 'RMSE', False), ('hit1', '1-hit', True),
                    ('hit2', '2-hit', True), ('hit3', '3-hit', True)]
    model_labels = model_labels or {}
    headers = [model_labels.get(m, m) for m in models]
    fmt = lambda v: '--' if pd.isna(v) else f'{v:.{decimals}f}'

    out = [r'\begin{table}[htb!]', r'\renewcommand{\arraystretch}{1.1}', r'\centering',
           r'\begin{tabular}{c|c|' + 'c' * len(models) + '}', r'\hline',
           'Market & Measure & ' + ' & '.join(headers) + r' \\', r'\hline']
    for site in sites:
        for i, (col, header, higher) in enumerate(measures):
            vals = [round(summary.loc[(site, m), col], decimals) for m in models]
            finite = [v for v in vals if pd.notna(v)]
            best = (max(finite) if higher else min(finite)) if finite else None
            cells = [(r'\textbf{' + fmt(v) + '}') if (best is not None and v == best) else fmt(v)
                     for v in vals]
            lead = (r'\multirow{' + str(len(measures)) + '}*{' + site_fmt.format(site) + '} & ' + header
                    if i == 0 else ' & ' + header)
            out.append(lead + ' & ' + ' & '.join(cells) + r' \\')
        out.append(r'\hline')
    out += [r'\end{tabular}', r'\vspace{.4cm}', r'\caption{' + caption + '}',
            r'\label{' + label + '}', r'\end{table}']
    return '\n'.join(out)


MODEL_LABELS = {'IMRCS_EM_K10': 'IMRCS', 'SMNL_IMPATIENCE_K10': 'SMNL-Imp'}
latex_table = per_site_to_latex(per_site_summary, SHOW_MODELS, model_labels=MODEL_LABELS)
(RECORD / 'expedia_imrcs_per_site_table.tex').write_text(latex_table)
print(latex_table)

In [ ]:
def per_site_nll_to_latex(summary, models, sites=SITE_ORDER, measures=None,
                          model_labels=None,
                          caption='A comparison of the predictive performance of the fitted choice models on the Expedia data',
                          label='Tab:compare_expedia_nll', site_fmt='Site {}'):
    """Like `per_site_to_latex`, but reports integer negative log-likelihood (testNLL)
    in place of RMSE. `measures` is a list of (column, header, higher_is_better, decimals),
    so each measure carries its own precision (NLL to 0 decimals, hit rates to 4)."""
    if measures is None:
        measures = [('testNLL', 'NLL', False, 0), ('hit1', '1-hit', True, 4),
                    ('hit2', '2-hit', True, 4), ('hit3', '3-hit', True, 4)]
    model_labels = model_labels or {}
    headers = [model_labels.get(m, m) for m in models]
    fmt = lambda v, d: '--' if pd.isna(v) else f'{v:.{d}f}'

    out = [r'\begin{table}[htb!]', r'\renewcommand{\arraystretch}{1.1}', r'\centering',
           r'\begin{tabular}{c|c|' + 'c' * len(models) + '}', r'\hline',
           'Market & Measure & ' + ' & '.join(headers) + r' \\', r'\hline']
    for site in sites:
        for i, (col, header, higher, dec) in enumerate(measures):
            vals = [round(summary.loc[(site, m), col], dec) for m in models]
            finite = [v for v in vals if pd.notna(v)]
            best = (max(finite) if higher else min(finite)) if finite else None
            cells = [(r'\textbf{' + fmt(v, dec) + '}') if (best is not None and v == best) else fmt(v, dec)
                     for v in vals]
            lead = (r'\multirow{' + str(len(measures)) + '}*{' + site_fmt.format(site) + '} & ' + header
                    if i == 0 else ' & ' + header)
            out.append(lead + ' & ' + ' & '.join(cells) + r' \\')
        out.append(r'\hline')
    out += [r'\end{tabular}', r'\vspace{.4cm}', r'\caption{' + caption + '}',
            r'\label{' + label + '}', r'\end{table}']
    return '\n'.join(out)

MODEL_LABELS = {'IMRCS_EM_K10': 'IMC', 'SMNL_IMPATIENCE_K10': 'IML'}
nll_latex_table = per_site_nll_to_latex(per_site_summary, SHOW_MODELS, model_labels=MODEL_LABELS)
(RECORD / 'expedia_imrcs_per_site_nll_table.tex').write_text(nll_latex_table)
print(nll_latex_table)